In [1]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

sys.path.append(str(Path.cwd().parent))

from src.data_pipeline import build_dataloaders

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

RAW_DIR = "../data/raw/chest_xray"

loaders = build_dataloaders(
    raw_dir=RAW_DIR,
    batch_size=32,
    val_size=0.15,
    num_workers=2
)

test_loader = loaders.test

print("Device:", device)
print("Test images:", len(test_loader.dataset))

Device: cuda
Test images: 624


In [3]:
class BaselineCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [4]:
baseline_model = BaselineCNN()

baseline_model.load_state_dict(
    torch.load(
        "../models/baseline_cnn.pth",
        map_location=device
    )
)

baseline_model = baseline_model.to(device)
baseline_model.eval()

print("Baseline CNN loaded.")

Baseline CNN loaded.


In [5]:
efficientnet_model = models.efficientnet_b0(
    weights=None
)

efficientnet_model.classifier[1] = nn.Linear(
    efficientnet_model.classifier[1].in_features,
    2
)

efficientnet_model.load_state_dict(
    torch.load(
        "../models/efficientnet_b0_transfer.pth",
        map_location=device
    )
)

efficientnet_model = efficientnet_model.to(device)
efficientnet_model.eval()

print("EfficientNet-B0 loaded.")

EfficientNet-B0 loaded.


In [6]:
def predict(model, loader):

    true_labels = []
    predictions = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            predicted = torch.argmax(outputs, dim=1)

            true_labels.extend(labels.numpy())
            predictions.extend(
                predicted.cpu().numpy()
            )

    return true_labels, predictions

In [7]:
baseline_true, baseline_pred = predict(
    baseline_model,
    test_loader
)

efficientnet_true, efficientnet_pred = predict(
    efficientnet_model,
    test_loader
)

print("Predictions completed.")

Predictions completed.


In [8]:
def get_metrics(true, pred):

    accuracy = accuracy_score(true, pred)

    precision = precision_score(
        true,
        pred,
        zero_division=0
    )

    recall = recall_score(
        true,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        true,
        pred,
        zero_division=0
    )

    return accuracy, precision, recall, f1

In [9]:
baseline_metrics = get_metrics(
    baseline_true,
    baseline_pred
)

print("Baseline CNN")
print("----------------")
print("Accuracy :", round(baseline_metrics[0], 4))
print("Precision:", round(baseline_metrics[1], 4))
print("Recall   :", round(baseline_metrics[2], 4))
print("F1 Score :", round(baseline_metrics[3], 4))

Baseline CNN
----------------
Accuracy : 0.851
Precision: 0.8235
Recall   : 0.9692
F1 Score : 0.8905


In [10]:
efficientnet_metrics = get_metrics(
    efficientnet_true,
    efficientnet_pred
)

print("EfficientNet-B0")
print("----------------")
print("Accuracy :", round(efficientnet_metrics[0], 4))
print("Precision:", round(efficientnet_metrics[1], 4))
print("Recall   :", round(efficientnet_metrics[2], 4))
print("F1 Score :", round(efficientnet_metrics[3], 4))

EfficientNet-B0
----------------
Accuracy : 0.8766
Precision: 0.8581
Recall   : 0.9615
F1 Score : 0.9069


In [11]:
cm_baseline = confusion_matrix(
    baseline_true,
    baseline_pred
)

cm_efficientnet = confusion_matrix(
    efficientnet_true,
    efficientnet_pred
)

print("Baseline CNN:")
print(cm_baseline)

print("\nEfficientNet-B0:")
print(cm_efficientnet)

Baseline CNN:
[[153  81]
 [ 12 378]]

EfficientNet-B0:
[[172  62]
 [ 15 375]]


In [12]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Baseline CNN",
        "EfficientNet-B0"
    ],

    "Accuracy": [
        baseline_metrics[0],
        efficientnet_metrics[0]
    ],

    "Precision": [
        baseline_metrics[1],
        efficientnet_metrics[1]
    ],

    "Recall": [
        baseline_metrics[2],
        efficientnet_metrics[2]
    ],

    "F1 Score": [
        baseline_metrics[3],
        efficientnet_metrics[3]
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,Baseline CNN,0.850962,0.823529,0.969231,0.890459
1,EfficientNet-B0,0.876603,0.858124,0.961538,0.906892
